In [ ]:

import os
import pandas as pd 
import os

from plotting import trace_plots as p_trace
from conversions import trace_conversions as c_trace

from analysis_util import *

from common_constants import *
import filters.trace_filters as f_trace
import plotly.io as pio

pio.templates.default = "seaborn"

update_figures =  False
images_folder = "../images"


dataset_path = f"../dataset/benchmark_parsed/conversion/run_0" # Dataset path for the benchmark_parsed dataset. This is where the parquet files are located.
query_path = os.path.join(dataset_path, f"dataset_query_search.parquet") # Path to the parquet file containing the query search data.
trace_path = os.path.join(dataset_path, f"dataset_trace_actions.parquet") # Path to the parquet file containing the trace actions data.
conv_trace_path = os.path.join(dataset_path, f"dataset_conv_trace_actions.parquet") # Path to the parquet file containing the converted trace actions data.
hashes_path = os.path.join(dataset_path, 'hashed_traces') # Path to the directory containing the hashed traces. 

df_query_search = pd.read_parquet(query_path) # Lookup table for trace group and trace identifiers, and the hash of the trace
df_query_search_copy = df_query_search.copy()
df_actions = pd.read_parquet(trace_path)
df_conv_actions = pd.read_parquet(conv_trace_path).merge(df_query_search[['trace_group_identifier', 'trace_identifier', 'hash_trace']], on='hash_trace', how='left')


In [2]:
import datetime
import time
common_limits = [
    0,
    10,
    25,
    50,
    75,
]

limit_point_thresholds = [
    c_trace.PointThreshold(0, 1),
    c_trace.PointThreshold(10, 0.75),
    c_trace.PointThreshold(25, 0.50),
    c_trace.PointThreshold(50, 0.25),
    c_trace.PointThreshold(75, 0.0)
]

      

In [ ]:

from common_constants import TraceConstants as tc

tgs: list[TraceGroupData] = []
# Process each trace group and generate the corresponding data.
print("SIM")
tgs += [process_tg(df_query_search, tc.tg_s_sim.value, f_trace.TRACE_FILTER_SYNTH_SIM, common_limits, limit_point_thresholds, hashes_path)]
print("RANDOM_EVEN")
tgs += [process_tg(df_query_search, tc.tg_s_random_even.value, f_trace.TRACE_FILTER_SYNTH_RANDOM_EVEN, common_limits, limit_point_thresholds, hashes_path)]
print("STACK")
tgs += [process_tg(df_query_search, tc.tg_s_stack.value, f_trace.TRACE_FILTER_SYNTH_STACK, common_limits, limit_point_thresholds, hashes_path)]
print("LARSON")
tgs += [process_tg(df_query_search, tc.tg_p_larson.value, f_trace.TRACE_FILTER_PROGRAM_LARSON, common_limits, limit_point_thresholds, hashes_path)]
print("CFRAC")
tgs += [process_tg(df_query_search, tc.tg_p_cfrac.value, f_trace.TRACE_FILTER_PROGRAM_CFRAC, common_limits, limit_point_thresholds, hashes_path)]

trace_groups = ['trace_group_identifier', 'trace_identifier']
model_groups = ['trace_group_identifier', 'trace_identifier', 'model_identifier']

df_trace_info = pd.concat([tg.df_trace_view for tg in tgs])
df_model_mems = pd.concat([tg.df_model_mem for tg in tgs])
df_model_frag = pd.concat([tg.df_model_frag for tg in tgs])
df_model_points = pd.concat([tg.df_model_points for tg in tgs])


SIM
RANDOM_EVEN
STACK
LARSON
CFRAC


Overview of traces

In [ ]:
# import plotly.io as pio
# pio.renderers.default = "notebook_connected"
import plotly.io as pio
pio.defaults.mathjax = None

# For each trace in a trace group, plot the trace, and save it to pdf
for tg in tgs:
    for i in range(0, 3):
        fig = p_trace.plot_trace_action_filtered(df_query_search,  df_conv_actions, title = f"", y_col="Bytes Allocated", show_legend=False, labels=PLOT_RENAME_MAPPING, wanted_traces=[t for t in tg.trace_identifiers if str(i) in t])
        fig.update_layout(height=300, width=700)
        if update_figures:
            save_fig_to_pdf(fig, f"{images_folder}/{tg.tg_identifier}_{i}.pdf")
    


# Overview of internal fragmentation of the trace and the trace groups

In [ ]:
import plotly.express as px
metrics_int = [
    IntFragConstants.relative.value,
    IntFragConstants.WJ_max.value,
    IntFragConstants.WJ_reserved_upper.value,
]

metrics_ext = [
    ExtFragConstants.Relative_PV.value,
    ExtFragConstants.LFB_PV.value,
    ExtFragConstants.WJ_alloc_reserved_upper.value,
    ExtFragConstants.WJ_req_reserved_upper.value,
    ExtFragConstants.WJ_alloc_max.value,
    ExtFragConstants.WJ_req_max.value,
]


METRICS_FILTER_T = [
    'model_identifier',
    'trace_group_identifier',
    'trace_identifier',
    'model_variables_allocator_identifier',
    'model_config_size_multiple',
]

METRIC_FILTER_TG = [
    'model_identifier',
    'trace_group_identifier',
    'model_variables_allocator_identifier',
    'model_config_size_multiple',
]

METRIC_FILTER_TOTAL = [
    'model_identifier',
    'model_variables_allocator_identifier',
    'model_config_size_multiple',
]

RENAME_SM = "Size mult"
RENAME_ALLOC_ID = 'Alloc ID'

print(f"Amt of total fragmentation metrics: {len(metrics_int + metrics_ext)}")
print(f"Amt of internal fragmentation metrics: {len(metrics_int)}")
print(f"Amt of external fragmentation metrics: {len(metrics_ext)}")

amt_tg = 5
traces_per_tg = 3
point_per_metric = 1

max_points_int_t = len(metrics_int) * point_per_metric
max_points_ext_t = len(metrics_ext) * point_per_metric
max_points_total_t = max_points_int_t + max_points_ext_t
max_points_int_tg = traces_per_tg * max_points_int_t
max_points_ext_tg = traces_per_tg * max_points_ext_t
max_points_total_tg = max_points_int_tg + max_points_ext_tg
max_points_int = max_points_int_tg * amt_tg
max_points_ext = max_points_ext_tg * amt_tg
max_points_total = max_points_int + max_points_ext


df_points = df_model_points[df_model_points.columns.intersection([*metrics_int, *metrics_ext, *METRICS_FILTER_T])]
df_points_int = df_model_points[df_model_points.columns.intersection([*metrics_int, *METRICS_FILTER_T])]
df_points_ext = df_model_points[df_model_points.columns.intersection([*metrics_ext, *METRICS_FILTER_T])]

# Points with size multiples, utility dataframes.
df_points_exsm = df_points.copy().drop('model_config_size_multiple', axis=1)
df_points_int_exsm = df_points_int.drop('model_config_size_multiple', axis=1)
df_points_ext_exsm = df_points_ext.drop('model_config_size_multiple', axis=1)

# Get all unique model identifiers and trace group identifiers for the points dataframes.
df_mf_model_identifier: pd.DataFrame = df_points[METRIC_FILTER_TOTAL].drop_duplicates()
df_mf_tg: pd.DataFrame = df_points[METRICS_FILTER_T].drop_duplicates()

# Total amount of points scored, grouped by model identifier, and sorted by points scored.
df_points_total = df_points_exsm.groupby(['model_identifier']).sum(numeric_only=True).sum(numeric_only=True, axis=1).to_frame("points_total").sort_values(ascending=False, by='points_total')
df_points_total_int = df_points_int_exsm.groupby(['model_identifier']).sum(numeric_only=True).sum(numeric_only=True, axis=1).to_frame("points_int").sort_values(ascending=False, by='points_int')
df_points_total_ext = df_points_ext_exsm.groupby(['model_identifier']).sum(numeric_only=True).sum(numeric_only=True, axis=1).to_frame("points_ext").sort_values(ascending=False, by='points_ext')

# Total amount of ideal points scored, grouped by model identifier, and sorted by points scored.
df_points_ideal_total = df_points_total.apply(lambda row: row['points_total']- max_points_total, axis=1).to_frame("ideal_total")
df_points_ideal_int = df_points_total_int.apply(lambda row: row['points_int'] - max_points_int, axis=1).to_frame("ideal_int")
df_points_ideal_ext = df_points_total_ext.apply(lambda row: row['points_ext'] - max_points_ext, axis=1).to_frame("ideal_ext")

# Total amount of ideal points scored, as a percentage of the maximum points scored, grouped by model identifier, and sorted by points scored.
df_points_ideal_total_perc = df_points_total.apply(lambda row: (row['points_total']- max_points_total)*100/max_points_total, axis=1).to_frame("ideal_total_perc")
df_points_ideal_int_perc = df_points_total_int.apply(lambda row: (row['points_int'] - max_points_int)*100/max_points_int, axis=1).to_frame("ideal_int_perc")
df_points_ideal_ext_perc = df_points_total_ext.apply(lambda row: (row['points_ext'] - max_points_ext)*100/max_points_ext, axis=1).to_frame("ideal_ext_perc")

# Points grouped by trace group and model identifier, and sorted by points scored. 
df_points_tgs = df_points_exsm.groupby(['trace_group_identifier', 'model_identifier']).sum(numeric_only=True).sum(numeric_only=True, axis=1).to_frame("points_total").sort_values(ascending=False, by=['trace_group_identifier', 'points_total', 'model_identifier'])
df_points_tgs_int = df_points_int_exsm.groupby(['trace_group_identifier','model_identifier']).sum(numeric_only=True).sum(numeric_only=True, axis=1).to_frame("points_int").sort_values(ascending=False, by=['trace_group_identifier', 'points_int', 'model_identifier'])
df_points_tgs_ext = df_points_ext_exsm.groupby(['trace_group_identifier','model_identifier']).sum(numeric_only=True).sum(numeric_only=True, axis=1).to_frame("points_ext").sort_values(ascending=False, by=['trace_group_identifier', 'points_ext', 'model_identifier'])

df_points_tgs_overview = pd.merge(df_points_tgs, df_points_tgs_int, on=['trace_group_identifier', 'model_identifier']).merge(df_points_tgs_ext, on=['trace_group_identifier', 'model_identifier'])

df_points_overview_points = df_points_total.merge(df_points_total_int, on='model_identifier').merge(df_points_total_ext, on='model_identifier')
df_points_overview_points = df_points_overview_points.merge(df_mf_model_identifier, on='model_identifier')[[
    *METRIC_FILTER_TOTAL,
    *[e for e in df_points_overview_points.columns.to_list() if e not in METRIC_FILTER_TOTAL]
]]

# Ideal points overview
df_points_overview_ideal = df_points_ideal_total.merge(df_points_ideal_int, on='model_identifier').merge(df_points_ideal_ext, on='model_identifier')
df_points_overview_ideal = df_points_overview_ideal.merge(df_mf_model_identifier, on='model_identifier')[[
    *METRIC_FILTER_TOTAL,
    *[e for e in df_points_overview_ideal.columns.to_list() if e not in METRIC_FILTER_TOTAL]
]]

# Ideal points overview as a percentage of the maximum points scored.
df_points_overview_ideal_perc = df_points_ideal_total_perc.merge(df_points_ideal_int_perc, on='model_identifier').merge(df_points_ideal_ext_perc, on='model_identifier')
df_points_overview_ideal_perc = df_points_overview_ideal_perc.merge(df_mf_model_identifier, on='model_identifier')[[
    *METRIC_FILTER_TOTAL,
    *[e for e in df_points_overview_ideal_perc.columns.to_list() if e not in METRIC_FILTER_TOTAL]
]]

df_points_overview_total = pd.merge(df_points_overview_points, df_points_overview_ideal, on=METRIC_FILTER_TOTAL).merge(df_points_overview_ideal_perc, on=METRIC_FILTER_TOTAL)

# Merge together the total points, internal points, and external points dataframes, and keep only the relevant columns.
df_3x3_total = df_points_total.merge(df_points_total_int, on=['model_identifier']).merge(df_points_total_ext, on=['model_identifier']).merge(df_mf_model_identifier, on=['model_identifier'])
df_3x3_total = df_3x3_total[[
    *METRIC_FILTER_TOTAL,
    *[e for e in df_3x3_total.columns.to_list() if e not in METRIC_FILTER_TOTAL]
]]

POINTS_OPTIONS = [
    'points_total',
    'points_int',
    'points_ext'
]


df_3x3_total = df_3x3_total.melt(
    id_vars=[c for c in df_3x3_total.columns.to_list() if c not in POINTS_OPTIONS],
    value_vars=POINTS_OPTIONS,
    var_name='point_type',
    value_name='point_value'
)
df_3x3_total['model_config_size_multiple'] = df_3x3_total['model_config_size_multiple'].apply(lambda x: str(x))


POINTS_OPTIONS = [
    'points_total',
    'points_int',
    'points_ext'
]

# Create a new dataframe for the total points analysis. Group by model and tracegroup, and sum the total points metrics.
# Then merge with the model identifier dataframe to get the allocator and size multiple information back.
df_3x3_total_mixed = df_points_tgs_overview.merge(df_mf_tg, on=['trace_group_identifier','model_identifier'])
df_3x3_total_mixed = df_3x3_total_mixed[[
    *METRIC_FILTER_TG,
    *[e for e in df_3x3_total_mixed.columns.to_list() if e not in METRIC_FILTER_TG]
]]
df_3x3_total_mixed = df_3x3_total_mixed.melt(
    id_vars=[c for c in df_3x3_total_mixed.columns.to_list() if c not in POINTS_OPTIONS],
    value_vars=POINTS_OPTIONS,
    var_name='point_type',
    value_name='point_value'
)
df_3x3_total_mixed['model_config_size_multiple'] = df_3x3_total_mixed['model_config_size_multiple'].apply(lambda x: str(x))

# Create a 3x3 plot of the total points scored, grouped by model identifier, and sorted by points scored.
fig_3x3_total_mixed = px.strip(df_3x3_total_mixed.copy(), 
                               labels=PLOT_RENAME_MAPPING, 
                               x='model_variables_allocator_identifier', 
                               y='point_value', 
                               facet_col='point_type',
                               facet_row='trace_group_identifier',  
                               title="", 
                               color='model_config_size_multiple')
fig_3x3_total_mixed.update_layout(width=800, height=700)
fig_3x3_total_mixed.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_3x3_total_mixed.for_each_annotation(lambda a: a.update(text=PLOT_RENAME_MAPPING.get(a.text)))

# Overall Findings, portable to latex. Summariz the points scored for total, internal and external fragmentation.
print(df_points_overview_total.to_string()) # Handy for overview, but too big for analysis
latex_copy: pd.DataFrame = df_points_overview_total.copy()
float_cols = latex_copy.select_dtypes(include=['float']).columns
latex_version = latex_copy.style \
                        .format("{:.1f}", subset=float_cols, escape="latex") \
                        .relabel_index(latex_copy.columns.map(lambda x: PLOT_RENAME_MAPPING[x]), axis=1) \
                        .hide(["model_identifier"], axis=1) \
                        .hide(axis="index") \
                        .to_latex(hrules=True)
print("\n" + latex_version)

fig_3x3_total_mixed.show()

if update_figures:
    save_fig_to_pdf(fig_3x3_total_mixed, f"{images_folder}/total_points.pdf")


Amt of total fragmentation metrics: 9
Amt of internal fragmentation metrics: 3
Amt of external fragmentation metrics: 6
                       model_identifier model_variables_allocator_identifier  model_config_size_multiple  points_total  points_int  points_ext  ideal_total  ideal_int  ideal_ext  ideal_total_perc  ideal_int_perc  ideal_ext_perc
0   merged-model_0_FFAO-1-size-multiple                                 FFAO                           1    116.691271   45.000000   71.691271   -18.308729   0.000000 -18.308729        -13.562022        0.000000      -20.343033
1  merged-model_1_FFAO-16-size-multiple                                 FFAO                          16    108.684162   39.881364   68.802798   -26.315838  -5.118636 -21.197202        -19.493213      -11.374746      -23.552447
2   merged-model_6_BFAO-1-size-multiple                                 BFAO                           1    107.211473   45.000000   62.211473   -27.788527   0.000000 -27.788527        -20.584094 

In [ ]:
import plotly.express as px

# Create a 3x3 version of only the internal fragmentation points.
df_3x3_total_int = df_points_tgs_int.merge(df_mf_tg, on=['trace_group_identifier','model_identifier'])
df_3x3_total_int = df_3x3_total_int[[
    *METRICS_FILTER_T,
    *[e for e in df_3x3_total_int.columns.to_list() if e not in METRICS_FILTER_T]
]]
df_3x3_total_int['model_config_size_multiple'] = df_3x3_total_int['model_config_size_multiple'].apply(lambda x: str(x))

# Get the internal fragmentation points for the synth_stack and program_cfrac trace groups.
df_3x3_int_frag_synth_cfrac =  df_points_int[df_points_int['trace_group_identifier'].isin(['synth_stack', 'program_cfrac'])].copy()
df_3x3_int_frag_synth_cfrac['model_config_size_multiple'] = df_3x3_int_frag_synth_cfrac['model_config_size_multiple'].apply(lambda x: str(x))
df_3x3_int_frag_synth_cfrac = df_3x3_int_frag_synth_cfrac.melt(
    id_vars=[c for c in df_3x3_int_frag_synth_cfrac.columns.to_list() if c not in metrics_int],
    value_vars=metrics_int,
    var_name='frag_type',
    value_name='points_int'
)

# Create a 3x3 plot of the internal fragmentation points for the synth_stack and program_cfrac trace groups.
fig_3x3_int_frag_synth_cfrac = px.strip(df_3x3_int_frag_synth_cfrac, labels=PLOT_RENAME_MAPPING, x='model_variables_allocator_identifier', y='points_int', facet_row='trace_group_identifier', facet_col='frag_type',  title="", color='model_config_size_multiple')
fig_3x3_int_frag_synth_cfrac.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_3x3_int_frag_synth_cfrac.for_each_annotation(lambda a: a.update(text=PLOT_RENAME_MAPPING.get(a.text)))
fig_3x3_int_frag_synth_cfrac.update_layout(width=800, height=400)

df_3x3_int_frag = df_points_int_exsm.groupby(['model_identifier', 'trace_group_identifier']).sum(numeric_only=True).copy().reset_index()
df_3x3_int_frag = df_3x3_int_frag.groupby(['model_identifier', 'trace_group_identifier']).sum(numeric_only=True).reset_index()
df_3x3_int_frag = df_3x3_int_frag.merge(df_mf_model_identifier, on=['model_identifier'])[[
    *METRIC_FILTER_TOTAL,
    *[e for e in df_3x3_int_frag.columns.to_list() if e not in METRIC_FILTER_TOTAL]
]]
df_3x3_int_frag['model_config_size_multiple'] = df_3x3_int_frag['model_config_size_multiple'].apply(lambda x: str(x))
df_3x3_int_frag = df_3x3_int_frag.melt(
    id_vars=[c for c in df_3x3_int_frag.columns.to_list() if c not in metrics_int],
    value_vars=metrics_int,
    var_name='frag_type',
    value_name='points_int'
)

fig_3x3_int_frag = px.strip(df_3x3_int_frag, labels=PLOT_RENAME_MAPPING, x='model_variables_allocator_identifier', y='points_int', facet_row='trace_group_identifier', facet_col='frag_type',  color='model_config_size_multiple')
fig_3x3_int_frag.update_layout(width=800, height=700)
fig_3x3_int_frag.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_3x3_int_frag.for_each_annotation(lambda a: a.update(text=PLOT_RENAME_MAPPING.get(a.text)))

import filters.trace_filters as t_f
import filters.model_filters as m_f
import filters.query_memory_usage_filters as q_f_mu
import filters.query_int_frag_filters as q_f_int
filter = t_f.get_tg(['synth_stack']) \
    .merge([t_f.get_trace_num(["0"])]) \
    .merge([m_f.MODEL_FILTER_POLICY_FFAO]) \
    .merge([q_f_int.QUERY_FILTER_FRAG_INT_RELATIVE]) \
    .merge([m_f.MODEL_FILTER_SIZE_MULT_16, m_f.MODEL_FILTER_SIZE_MULT_64]) \

filter2 = t_f.get_tg(['synth_stack']) \
    .merge([t_f.get_trace_num(["0"])]) \
    .merge([m_f.MODEL_FILTER_POLICY_FFAO]) \
    .merge([q_f_int.QUERY_FILTER_FRAG_INT_RELATIVE]) \
    .merge([m_f.MODEL_FILTER_SIZE_MULT_1]) \

df_trace_1 = filter.apply(df_query_search_copy)
df_trace_2 = filter2.apply(df_query_search_copy)
df_trace = pd.concat([df_trace_1, df_trace_2])

import plotting.plot_util as p_util
import conversions.conversion_util as cu
df_query_data = cu.get_query_data(df_trace, hashes_path)
print(df_query_data['query_identifier'].unique().tolist())
print(df_query_data['model_config_size_multiple'].unique().tolist())
fig = p_util.show_index_chart_query_data_upsampled("Relative Internal Fragmentation FFAO: Stack Trace #0", df_query_data, y_name='Frag. %', labels=PLOT_RENAME_MAPPING, port=get_free_port())
fig.update_traces(marker=dict(size=2))
fig.update_layout(legend=dict(
    orientation="h",      
        yanchor="bottom",     
        y=1.02,               
        xanchor="right",      
        x=1                
))
colors = [
    "#1a9850",
    "#91cf60",
    "#ffffbf",
    "#d88b17",
    "#d73027" 
]

# Add horizontal lines and rectangles to the plot to indicate the common limits and their corresponding weights.
for i, lim in enumerate(common_limits):
    opacity=0.20
    # Create a horizontal line between limits, and a rectangle to indicate the weight of the limit.
    if i == 0:
        pos = common_limits[i+1] / 2
        fig.add_hline(y=pos, line_width=1, annotation_position="right", line_color="rgba(0,0,0,0)",  annotation_text=f"Weight: {limit_point_thresholds[i][1]}")
    elif i < len(common_limits) - 1:
        pos = (common_limits[i+1] - lim) / 2 + lim
        fig.add_hline(y=pos, line_width=1, annotation_position="right", line_color="rgba(0,0,0,0)",  annotation_text=f"Weight: {limit_point_thresholds[i][1]}")
        fig.add_hrect(
        y0=common_limits[i-1], y1=lim, 
        fillcolor=colors[i-1], # Choose the associated color based on the common limits.
        opacity=opacity, 
        line_width=0)
    # Create a last horizontal line and rectangle to indicate the weight of the last limit.
    else:
        max_y = max([max(trace.y) for trace in fig.data if trace.y is not None])
        pos = (100 - lim) / 2 + lim
        fig.add_hline(y=pos, line_width=1, annotation_position="right", line_color="rgba(0,0,0,0)", annotation_text=f"Weight: {limit_point_thresholds[i][1]}")
        fig.add_hrect(
        y0=common_limits[i-1], y1=lim, 
        fillcolor=colors[i-1], 
        opacity=opacity, 
        line_width=0)

        fig.add_hrect(
        y0=lim, y1=100, 
        fillcolor=colors[i], 
        opacity=opacity, 
        line_width=0)



fig.show()
fig.update_layout(height=300, width=700)

fig_3x3_int_frag.show()
fig_3x3_int_frag_synth_cfrac.show()

if update_figures:
    save_fig_to_pdf(fig_3x3_int_frag, f"{images_folder}/int_frag.pdf")
    save_fig_to_pdf(fig_3x3_int_frag_synth_cfrac, f"{images_folder}/int_frag_traceview.pdf")
    save_fig_to_pdf(fig, f"{images_folder}/weights.pdf")



['frag_int_relative']
[1, 16, 64]


In [ ]:
# Displays the external fragmentation analysis in a 3x3 plot.

# Create a new dataframe for the external fragmentation analysis. Group by model and tracegroup, and sum the external fragmentation metrics. 
# Then merge with the model identifier dataframe to get the allocator and size multiple information back.
df_3x3_ext_frag = df_points_ext_exsm.groupby(['model_identifier', 'trace_group_identifier']).sum(numeric_only=True).copy().reset_index()
df_3x3_ext_frag = df_3x3_ext_frag.groupby(['model_identifier', 'trace_group_identifier']).sum().reset_index()
df_3x3_ext_frag = df_3x3_ext_frag.merge(df_mf_model_identifier, on=['model_identifier'])[[
    *METRIC_FILTER_TOTAL,
    *[e for e in df_3x3_ext_frag.columns.to_list() if e not in METRIC_FILTER_TOTAL]
]]

# Melt the dataframe to have a long format, with one row per metric type and one column for the metric value.
df_3x3_ext_frag = df_3x3_ext_frag.melt(
    id_vars=[c for c in df_3x3_ext_frag.columns.to_list() if c not in metrics_ext],
    value_vars=metrics_ext,
    var_name='frag_type',
    value_name='points_ext'
)
df_3x3_ext_frag['points_ext'] = df_3x3_ext_frag['points_ext'].apply(lambda x: float(x))

fig_3x3_ext_frag = px.strip( df_3x3_ext_frag, 
                            labels=PLOT_RENAME_MAPPING, 
                            x='model_variables_allocator_identifier', 
                            y='points_ext', 
                            facet_row='trace_group_identifier', 
                            facet_col='frag_type',  
                            title="", 
                            color='model_config_size_multiple')

# Rename labels
fig_3x3_ext_frag.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_3x3_ext_frag.for_each_annotation(lambda a: a.update(text=PLOT_RENAME_MAPPING.get(a.text)))
fig_3x3_ext_frag.update_layout(width=1400, height=700)
fig_3x3_ext_frag.show()

if update_figures:
    save_fig_to_pdf(fig_3x3_ext_frag, f"{images_folder}/ext_frag.pdf")

